# AI 2 · 3주차 회전 확인 개선 검증 (30분)

3주차 실험에서 **콜라 캔은 원재료 표가 세로로 인쇄**되어 있어서, 0°와 180°만 확인하던 회전 보정으로는 잘 읽히지 않았습니다. 그래서 회전 확인 후보에 **90°와 270°**를 추가했습니다.

이 노트북은 회전·뒤집힘과 관련된 사진 10장만 다시 돌려서, 3주차 실험 결과와 비교합니다.

**실행 방법:** 메뉴에서 **런타임 → 모두 실행**. 약 30분 걸리고, 끝나면 `ai2_week3_orientation.zip`이 내려받아집니다.

In [ ]:
REPO_URL = "https://github.com/DL-2026F-Team05/package-ocr-preprocessing.git"
import os, subprocess, sys
os.chdir("/content")
if not os.path.exists("package-ocr"):
    if subprocess.run(["git", "clone", "-q", REPO_URL, "package-ocr"]).returncode != 0:
        raise RuntimeError("레포를 받아오지 못했습니다.")
os.chdir("/content/package-ocr")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "paddlepaddle", "paddleocr"], check=True)
print("준비 완료:", os.getcwd())

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, "preprocessing")
import experiment, preprocess, pipeline
from ocr_eval import OCREngine, region_cer

FILES = ["p01_c6_L3", "p01_c6_L4", "p02_c0_1", "p02_c6_2", "p03_c0_1", "p03_c6_1",
         "p05_c6_1", "p05_c6_2", "p07_c6_1", "p07_c6_2"]
RAW, GEN, OUT = Path("data/raw"), Path("data/generated"), Path("results/ai2_week3/orientation")
OUT.mkdir(parents=True, exist_ok=True)
experiment.generate_from_clean(RAW, GEN, conditions=("c6",), products=["p01"])
labels = experiment.load_labels("data/labels/labels.csv")
ocr = OCREngine("paddle")
print("OCR 준비 완료")

In [ ]:
rows = []
for n, name in enumerate(FILES, 1):
    path = RAW / f"{name}.jpg" if (RAW / f"{name}.jpg").exists() else GEN / f"{name}.jpg"
    out = pipeline.run_pipeline(preprocess.load_image(path), ocr)
    text = " ".join(r["text"] for r in out["ocr"])
    lab = labels[name[:3]]
    cers = {f"cer_{k}": round(region_cer(text, v), 3) for k, v in lab.items()}
    rows.append({"file": f"{name}.jpg", "rotation": out["preprocess"]["rotation"],
                 "rotation_scores": out["preprocess"]["rotation_scores"],
                 "confidence": out["quality"]["ocr_confidence"], "retake": out["quality"]["retake"],
                 "sec": out["elapsed_sec"], **cers, "cer_mean": round(float(np.mean(list(cers.values()))), 4),
                 "ocr_text": text})
    print(f"[{n}/{len(FILES)}] {name}: 회전 {out['preprocess']['rotation']}°, 오류율 {rows[-1]['cer_mean']}, {out['elapsed_sec']}초")
new = pd.DataFrame(rows)
new.to_csv(OUT / "orientation_check.csv", index=False, encoding="utf-8-sig")

In [ ]:
old_path = Path("results/ai2_week3/paddle/ocr_results.csv")
if old_path.exists():
    old = pd.read_csv(old_path)
    old = old[old.variant == "auto"][["file", "rotation", "cer_mean"]].rename(columns={"rotation": "rotation_before", "cer_mean": "cer_before"})
    cmp = new[["file", "rotation", "cer_mean", "sec"]].merge(old, on="file")
    print("평균 오류율: 이전", round(cmp.cer_before.mean(), 3), "→ 지금", round(cmp.cer_mean.mean(), 3))
    display(cmp)
else:
    display(new[["file", "rotation", "cer_mean", "sec"]])

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/ai2_week3_orientation", "zip", OUT)
files.download("/content/ai2_week3_orientation.zip")